### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import evaluate
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
from transformers import TrainingArguments, Trainer

from data.kaggle_text import load_kaggle_disaster_csv
from models.text_branch import create_text_classifier


### Load Kaggle dataset and tokenize

In [ ]:
# Load Kaggle CSV as HF Dataset
dataset = load_kaggle_disaster_csv("../data/train.csv")  # adjust path

# Train/validation split (80/20)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

# Create model + tokenizer
model, tokenizer = create_text_classifier(num_labels=2)

# Tokenization function
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized = dataset.map(tokenize_batch, batched=True)

train_ds = tokenized["train"]
val_ds   = tokenized["test"]

# Tell Trainer which columns are inputs/labels
train_ds = train_ds.remove_columns(
    [c for c in train_ds.column_names if c not in ["input_ids", "attention_mask", "label"]]
)
val_ds = val_ds.remove_columns(
    [c for c in val_ds.column_names if c not in ["input_ids", "attention_mask", "label"]]
)

train_ds.set_format("torch")
val_ds.set_format("torch")


### Setup metrics and training

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)

output_dir = "../checkpoints/text_branch"

training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)


### Train the model

In [ ]:
train_result = trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

### Plot training/eval metrics

In [ ]:
# Extract history
logs = pd.DataFrame(trainer.state.log_history)

# Filter rows with evaluation metrics
eval_logs = logs[logs["event"] == "evaluation"].copy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax[0].plot(eval_logs["epoch"], eval_logs["eval_accuracy"], marker="o")
ax[0].set_title("Validation Accuracy")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Accuracy")

# Loss (if logged)
if "eval_loss" in eval_logs.columns:
    ax[1].plot(eval_logs["epoch"], eval_logs["eval_loss"], marker="o", color="orange")
    ax[1].set_title("Validation Loss")
    ax[1].set_xlabel("Epoch")
    ax[1].set_ylabel("Loss")

plt.tight_layout()
plt.show()

### Confusion matrix on validation set

In [ ]:
preds = trainer.predict(val_ds)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Not disaster (0)", "Disaster (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Blues", ax=ax, colorbar=False)
plt.title("DistilBERT Text Model - Confusion Matrix (Validation)", fontsize=14)
plt.grid(False)
plt.show()